# Wi-Fi Fingerprint Indoor Localization — Classification Track (Part A)

**Objective:** Predict which **floor** a device is on based on its Wi-Fi RSSI fingerprint.

**Dataset:** UJIndoorLoc (UCI ML Repository) — same dataset as regression track.

**Team:** K Ganesh Giridhar (519) · G R Balaji (510) · A Suhas Reddy (503)

---
## 1. Imports & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded")

In [ ]:
# Load data
train_df = pd.read_csv('../data/raw/trainingData.csv')
val_df   = pd.read_csv('../data/raw/validationData.csv')

wap_cols = [c for c in train_df.columns if c.startswith('WAP')]
print(f"Loaded {train_df.shape[0]} training samples, {len(wap_cols)} WAPs")

## 2. Preprocessing

We apply the same cleaning pipeline as the regression track to ensure consistency.

In [ ]:
# Replace sentinel +100 with -105
train_clean = train_df.copy()
train_clean[wap_cols] = train_clean[wap_cols].replace(100, -105)

# Remove zero-variance WAPs
variances = train_clean[wap_cols].var()
zero_var = variances[variances == 0].index.tolist()
wap_cols_clean = [c for c in wap_cols if c not in zero_var]
print(f"WAPs after removing zero-variance: {len(wap_cols_clean)}")

In [ ]:
# Feature engineering — same RSSI summary features
def engineer_features(df, wap_columns):
    wap_data = df[wap_columns]
    df['n_visible_aps'] = (wap_data > -105).sum(axis=1)
    detected = wap_data.replace(-105, np.nan)
    df['mean_rssi'] = detected.mean(axis=1)
    df['max_rssi'] = detected.max(axis=1)
    df['std_rssi'] = detected.std(axis=1)
    df['rssi_range'] = detected.max(axis=1) - detected.min(axis=1)
    return df

train_clean = engineer_features(train_clean, wap_cols_clean)

eng_features = ['n_visible_aps', 'mean_rssi', 'max_rssi', 'std_rssi', 'rssi_range']
feature_cols = wap_cols_clean + eng_features

print(f"Total features: {len(feature_cols)}")

### 2.1 Target Variable — Floor

In [ ]:
# Classification target
y = train_clean['FLOOR']
print("Floor distribution:")
print(y.value_counts().sort_index())
print()
print(f"Number of classes: {y.nunique()}")

In [ ]:
# Visualize class distribution
y.value_counts().sort_index().plot(kind='bar', color='seagreen', edgecolor='black')
plt.xlabel('Floor')
plt.ylabel('Number of Samples')
plt.title('Class Distribution — Floor')
plt.tight_layout()
plt.show()

**Observation:** Some floor imbalance exists but it's not extreme. We'll use stratified splitting to maintain class proportions.

### 2.2 Train-Test Split (Stratified)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = train_clean[feature_cols]

# Stratified split to preserve floor proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {X_train.shape[0]} | Testing: {X_test.shape[0]}")
print()
print("Train class distribution:")
print(y_train.value_counts().sort_index())
print()
print("Test class distribution:")
print(y_test.value_counts().sort_index())

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done (fitted on train only)")